In [2]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## Setup api key


In [3]:
import os 
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [9]:
import re 

# ----Deterministic approach to guardrails ----

def deterministic_guardrails(text:str)-> bool:
    """Return True if content is blocked"""
    banned_keywords = ["hack","exploit","malware","bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "what is capital of france?",
    "Explain how malware spreads."
]

print("=== Deterministic Guardrails Demo ===")

for inp in test_inputs:
    if deterministic_guardrails(inp):
        print(f"🚫 Blocked content: {inp}")
    else:
        print(f"✅ Allowed content: {inp}")


=== Deterministic Guardrails Demo ===
🚫 Blocked content: How do I hack into a database?
✅ Allowed content: what is capital of france?
🚫 Blocked content: Explain how malware spreads.


In [10]:
from langchain_openai import ChatOpenAI

def model_based_guardrails(text:str)-> bool:
    """Uses an LLM to evaluate content saftey. returns SAFE or UNSAFE."""
    model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
    prompt = f"""Is the following user input safe to process? 
    Reply with only 'SAFE' or 'UNSAFE'.
    
    Input: {text}"""
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("\n=== Model-based Guardrails Demo ===")
for inp in test_inputs:
    result = model_based_guardrails(inp)
    if result == "UNSAFE":
        print(f"🚫 Blocked content: {inp}")
    else:
        print(f"✅ Allowed content: {inp}")

    


=== Model-based Guardrails Demo ===
🚫 Blocked content: How do I hack into a database?
✅ Allowed content: what is capital of france?
✅ Allowed content: Explain how malware spreads.


### Section 3: Built in guardrail -- PII detection Middleware

Langchain provides built in PIIMiddleware for detecting and handling Personally identifiable information (PII)

Supported PII Types:

- email
- creditcard
- ip
- mac-address
- url

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Looks up customer information based on a query."""
    # Simulated database lookup
    return f"Customer record info found for query: {query}"

agent = create_agent(
    model="gpt-4o",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
        "email",strategy="redact",apply_to_input=True
        ),
        PIIMiddleware(
        "credit_card",strategy="mask",apply_to_input=True,
        ),
         PIIMiddleware(
        "api_key",detector=r"sk-[a-zA-Z0-9]{32}",strategy="block",apply_to_input=True,
        ),
    ]
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [18]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("==Agent Response==")
print(result["messages"][-1].content)

==Agent Response==
Yes, I can help you. Could you please provide more details about the issue you're facing or the assistance you need?


In [19]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='ddce39a7-47d4-4c68-9dc9-e0dfa29e6b58'),
  AIMessage(content="Yes, I can help you. Could you please provide more details about the issue you're facing or the assistance you need?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 74, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f89046aead', 'id': 'chatcmpl-Eifta7uIHRtDbqGPFlz9dVmghVYUu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': No

### Test API key blocking

In [20]:
try:
    result = agent.invoke({
        "messages":[{
            "role":"user",
            "content":"My API key is sk-1234567890abcdef1234567890abcdef. Can you help me?"
        }]
    })
except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


In [21]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='ddce39a7-47d4-4c68-9dc9-e0dfa29e6b58'),
  AIMessage(content="Yes, I can help you. Could you please provide more details about the issue you're facing or the assistance you need?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 74, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f89046aead', 'id': 'chatcmpl-Eifta7uIHRtDbqGPFlz9dVmghVYUu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': No

---
## 👤 Section 4: Built-in Guardrail — Human-in-the-Loop Middleware

Pauses agent execution before sensitive operations and waits for human approval.

**Best for:**
- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

**Key requirement:** A `checkpointer` for state persistence across interrupts.

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model="gpt-4o",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [35]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='378ee534-ea42-4497-a187-9ffc93b79359'), AIMessage(content='What would you like the subject and body of the email to be?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 109, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-Eignr6Qe1re5C02gdKWjdJf7KnJvZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1c5c9-6e07-7777-

In [34]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

=== Approved! Final response ===
What would you like the subject and body of the email to be?


In [36]:
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke({
    "messages":[{"role":"user","content":"Delete all records from the users table where active=false"}]
}, config=config2)



{'messages': [HumanMessage(content='Delete all records from the users table where active=false', additional_kwargs={}, response_metadata={}, id='6dbc1282-0084-4782-a869-3cd4fe762501'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 106, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EihInOWnjJxj8jbm6YxN33N9dTvB5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1c5e6-b063-7144-b967-a8a1301ddcd4-0', tool_calls=[{'name': 'delete_records', 'args': {'table': 'users', 'condit

In [37]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"reject",
                                  "reason":"Too risky, needs DBA review"
                                  }]}), config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===
It looks like deleting the records was stopped. If you have any questions or need further assistance, please let me know!


In [ ]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgenticMiddleware):
    """
        Deterministic guardrail: Block requests containing banned keywords.
        This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any]|None:
        if not state["messages"]:
            return None
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked - Keyword detected: {keyword}")
                return {
                                    "messages": [{
                                        "role": "assistant",
                                        "content": (
                                            "I cannot process requests containing inappropriate content. "
                                            "Please rephrase your request."
                                        )
                                    }],
                                    "jump_to": "end"
                                }
        return None

@tool
def search_tool(query:str)->str:
    """Search for information"""
    